# Pneumonia Detection — DenseNet121 vs ResNet50 (Comparative Study)

Dataset: Chest X-Ray Images (Pneumonia) — Kermany dataset, Google Drive er **Pneumonia** folder e rakha ache.
Ei notebook 2 ta model (DenseNet121 ar ResNet50) ek-i split e train kore compare kore —
accuracy, F1, ROC-AUC, PR-AUC, confusion matrix, training curve, metrics chart ar **Grad-CAM** shoho
sob figure **Pneumonia/Result** folder e save kore dey, paper e use korar jonno.

Sudhu **Runtime → Run all** chaplei hobe. (Age Runtime → Change runtime type → **T4 GPU** select kore nio.)


In [ ]:
# Google Drive mount kori
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, random, shutil, zipfile, time, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Drive e Pneumonia folder ta khuji (naam er seshe space thakleo pabe)
DRIVE_FOLDER = '/content/drive/MyDrive/Pneumonia'
for name in os.listdir('/content/drive/MyDrive'):
    if name.strip().lower() == 'pneumonia':
        DRIVE_FOLDER = os.path.join('/content/drive/MyDrive', name)
        break
print('Pneumonia folder:', DRIVE_FOLDER)

RESULT_FOLDER = os.path.join(DRIVE_FOLDER, 'Result')
os.makedirs(RESULT_FOLDER, exist_ok=True)
print('Result folder:', RESULT_FOLDER)


In [ ]:
# Dataset khuji - folder akare thakle direct, part kora zip thakle jora lagiye unzip kori
# Dataset er vitore train / test / val folder thakbe, protteke NORMAL ar PNEUMONIA subfolder
def find_dataset(base):
    for root, dirs, files in os.walk(base):
        if 'train' in dirs and 'test' in dirs:
            train_dir = os.path.join(root, 'train')
            if 'NORMAL' in os.listdir(train_dir) and 'PNEUMONIA' in os.listdir(train_dir):
                return root
    return None

data_root = find_dataset(DRIVE_FOLDER)

if data_root is None:
    # Dataset ta 3 ta part e vag kora ache (boro file bole) - age jora lagai, tarpor unzip
    parts = sorted(n for n in os.listdir(DRIVE_FOLDER) if n.startswith('chest_xray.zip.part-'))
    if parts:
        print('Dataset er part gula peyechi, jora lagai:', parts)
        joined = '/content/chest_xray.zip'
        with open(joined, 'wb') as out:
            for p in parts:
                with open(os.path.join(DRIVE_FOLDER, p), 'rb') as f:
                    shutil.copyfileobj(f, out)
        print('Zip jora lagano sesh, unzip kori')
        with zipfile.ZipFile(joined) as z:
            z.extractall('/content/pneumonia_data')
        data_root = find_dataset('/content/pneumonia_data')

if data_root is None:
    for name in os.listdir(DRIVE_FOLDER):
        if name.endswith('.zip'):
            print('Zip peyechi, unzip kori:', name)
            with zipfile.ZipFile(os.path.join(DRIVE_FOLDER, name)) as z:
                z.extractall('/content/pneumonia_data')
            data_root = find_dataset('/content/pneumonia_data')
            break

print('Dataset path:', data_root)

classes = ['NORMAL', 'PNEUMONIA']
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(data_root, split)
    if os.path.isdir(split_dir):
        for c in classes:
            n = len(os.listdir(os.path.join(split_dir, c)))
            print(split, '-', c, ':', n)


In [ ]:
# Train ar val folder mile train pool banai, seta theke seed 42 diye 12% validation alada kori
# Test set alada-i thake, oita sudhu seshe ekbar evaluate hobe (2 ta model er jonnoi ekbar kore)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
rng = random.Random(42)

train_items, val_items, test_items = [], [], []
for label, c in enumerate(classes):
    pool_files = []
    for split in ['train', 'val']:
        folder = os.path.join(data_root, split, c)
        if os.path.isdir(folder):
            pool_files += [os.path.join(folder, f) for f in sorted(os.listdir(folder))]
    rng.shuffle(pool_files)
    n_val = int(round(len(pool_files) * 0.12))
    val_items += [(f, label) for f in pool_files[:n_val]]
    train_items += [(f, label) for f in pool_files[n_val:]]

    folder = os.path.join(data_root, 'test', c)
    test_items += [(os.path.join(folder, f), label) for f in sorted(os.listdir(folder))]

print('Train:', len(train_items))
print('Validation:', len(val_items))
print('Test:', len(test_items))

# Class weight - NORMAL class kom ache, tai loss e weight diye balance kori
train_labels = np.array([y for _, y in train_items])
class_weights = torch.tensor(len(train_labels) / (2.0 * np.bincount(train_labels)),
                             dtype=torch.float32).to(device)
print('Class weights:', class_weights.cpu().numpy())


In [ ]:
# Image preprocessing + augmentation (augmentation sudhu training e)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class XrayDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

def make_loader(items, transform, batch_size, shuffle=False):
    return DataLoader(XrayDataset(items, transform), batch_size=batch_size,
                      shuffle=shuffle, num_workers=2)

val_loader = make_loader(val_items, test_transform, 64)
test_loader = make_loader(test_items, test_transform, 64)
print('Loaders ready')


In [ ]:
# Ekta loader er upor evaluate kori - accuracy, macro F1, confusion matrix,
# prottek image er prediction ar pneumonia probability o ferot dei (ROC/PR curve ar McNemar test er jonno)
def evaluate(model, loader):
    model.eval()
    cm = np.zeros((2, 2), dtype=int)
    all_pred, all_label, all_prob = [], [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            pred = logits.argmax(1).cpu().numpy()
            for t, p in zip(y.numpy(), pred):
                cm[t][p] += 1
            all_pred += pred.tolist()
            all_label += y.numpy().tolist()
            all_prob += prob.tolist()
    acc = np.trace(cm) / cm.sum()
    f1_list = []
    for i in range(2):
        tp = cm[i][i]
        precision = tp / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
        recall = tp / cm[i, :].sum() if cm[i, :].sum() > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_list.append(f1)
    return acc, float(np.mean(f1_list)), cm, np.array(all_pred), np.array(all_label), np.array(all_prob)


In [ ]:
# 2 ta model - 2 tai ImageNet pretrained, last layer 2 class er jonno change kori
def get_densenet121():
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    model.classifier = nn.Linear(model.classifier.in_features, 2)
    return model.to(device)

def get_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)

print('Model functions ready')


In [ ]:
# Training function - prottek epoch e validation dekhi, best validation F1 er model ta rekhe dei
def train_model(name, model, optimizer, scheduler, epochs, batch_size):
    train_loader = make_loader(train_items, train_transform, batch_size, shuffle=True)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = 0
    loss_history, val_f1_history = [], []

    for epoch in range(epochs):
        start = time.time()
        model.train()
        total_loss, total = 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            total += len(y)
        scheduler.step()

        val_acc, val_f1, _, _, _, _ = evaluate(model, val_loader)
        loss_history.append(total_loss / total)
        val_f1_history.append(val_f1)
        print(f'[{name}] Epoch {epoch+1}/{epochs}  loss={total_loss/total:.4f}  val_acc={val_acc*100:.2f}%  val_f1={val_f1*100:.2f}%  ({time.time()-start:.0f}s)')

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'best_{name}.pth')
            print('   -> best model saved')

    print(name, 'training sesh. Best validation F1:', round(best_f1 * 100, 2), '%')
    model.load_state_dict(torch.load(f'best_{name}.pth'))
    return loss_history, val_f1_history


In [ ]:
# Model 1: DenseNet121 - SGD optimizer (Hashmi et al. er recipe)
densenet = get_densenet121()
optimizer = torch.optim.SGD(densenet.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
dn_loss, dn_val_f1 = train_model('DenseNet121', densenet, optimizer, scheduler, epochs=25, batch_size=32)


In [ ]:
# Model 2: ResNet50 - Adam optimizer, proti 10 epoch e learning rate half
resnet = get_resnet50()
optimizer = torch.optim.Adam(resnet.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
rn_loss, rn_val_f1 = train_model('ResNet50', resnet, optimizer, scheduler, epochs=30, batch_size=16)


In [ ]:
# Final test - 2 ta model-i ek-i test set e, ekbar kore
from sklearn.metrics import roc_auc_score, average_precision_score

results = {}
for name, model in [('DenseNet121', densenet), ('ResNet50', resnet)]:
    acc, f1, cm, pred, label, prob = evaluate(model, test_loader)
    auc = roc_auc_score(label, prob)
    ap = average_precision_score(label, prob)
    results[name] = {'acc': acc, 'f1': f1, 'cm': cm, 'pred': pred,
                     'label': label, 'prob': prob, 'auc': auc, 'ap': ap}

    print('=' * 55)
    print(name)
    print('=' * 55)
    print(f'Accuracy : {acc * 100:.2f}%')
    print(f'Macro F1 : {f1 * 100:.2f}%')
    print(f'ROC-AUC  : {auc:.4f}')
    print(f'PR-AUC   : {ap:.4f}')
    for i, c in enumerate(classes):
        tp = cm[i][i]
        precision = tp / cm[:, i].sum() * 100
        recall = tp / cm[i, :].sum() * 100
        cf1 = 2 * precision * recall / (precision + recall)
        print(f'{c:10s} Precision={precision:.2f}  Recall={recall:.2f}  F1={cf1:.2f}  Total={cm[i, :].sum()}')
    tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    print(f'Sensitivity (pneumonia recall) : {tp / (tp + fn) * 100:.2f}%')
    print(f'Specificity (normal recall)    : {tn / (tn + fp) * 100:.2f}%')
    print('Confusion Matrix (row = asol class, column = prediction):')
    print(cm)
    print('Vul prediction:', int(cm.sum() - np.trace(cm)), '/', int(cm.sum()))
    print()


In [ ]:
# Confusion matrix er chobi - 2 ta model pasapasi (paper er jonno)
def draw_cm(ax, cm, title):
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(2))
    ax.set_yticks(range(2))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i][j], ha='center', va='center',
                    color='white' if cm[i][j] > cm.max() / 2 else 'black')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
draw_cm(axes[0], results['DenseNet121']['cm'], 'DenseNet121')
draw_cm(axes[1], results['ResNet50']['cm'], 'ResNet50')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'confusion_matrices.png'), dpi=300)
plt.show()

# Alada alada file o rakhi
for name in ['DenseNet121', 'ResNet50']:
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    draw_cm(ax, results[name]['cm'], name)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_FOLDER, f'confusion_matrix_{name}.png'), dpi=300)
    plt.show()


In [ ]:
# Training curve - 2 ta model er loss ar validation F1 ek-i graph e
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(dn_loss, label='DenseNet121')
axes[0].plot(rn_loss, label='ResNet50')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot([f * 100 for f in dn_val_f1], label='DenseNet121')
axes[1].plot([f * 100 for f in rn_val_f1], label='ResNet50')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Macro F1 %')
axes[1].set_title('Validation F1')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'training_curves.png'), dpi=300)
plt.show()


In [ ]:
# ROC curve ar Precision-Recall curve - 2 ta model ek-i graph e compare
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name in ['DenseNet121', 'ResNet50']:
    r = results[name]
    fpr, tpr, _ = roc_curve(r['label'], r['prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {r['auc']:.4f})")
    precision, recall, _ = precision_recall_curve(r['label'], r['prob'])
    axes[1].plot(recall, precision, label=f"{name} (AP = {r['ap']:.4f})")

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'roc_pr_curves.png'), dpi=300)
plt.show()


In [ ]:
# Metrics comparison bar chart - paper er main comparison figure
metrics_names = ['Accuracy', 'Macro F1', 'Sensitivity', 'Specificity', 'ROC-AUC', 'PR-AUC']
chart_data = {}
for name in ['DenseNet121', 'ResNet50']:
    r = results[name]
    cm = r['cm']
    tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    chart_data[name] = [r['acc'] * 100, r['f1'] * 100,
                        tp / (tp + fn) * 100, tn / (tn + fp) * 100,
                        r['auc'] * 100, r['ap'] * 100]

x = np.arange(len(metrics_names))
width = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width / 2, chart_data['DenseNet121'], width, label='DenseNet121')
bars2 = ax.bar(x + width / 2, chart_data['ResNet50'], width, label='ResNet50')
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.7,
                f'{bar.get_height():.1f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylabel('%')
ax.set_ylim(0, 112)
ax.set_title('DenseNet121 vs ResNet50 - Test Set Metrics')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'metrics_comparison.png'), dpi=300)
plt.show()


In [ ]:
# Grad-CAM: model image er kon jayga dekhe decision nilo seta heatmap diye dekhai
# Last convolution block er output theke activation + gradient niye heatmap banano hoy
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, x, class_idx):
        self.model.zero_grad()
        output = self.model(x)
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = torch.relu(cam)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

def cam_overlay(model, target_layer, image):
    gradcam = GradCAM(model, target_layer)
    model.eval()
    x = test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    pred = int(probs.argmax())
    conf = float(probs[pred]) * 100
    cam = gradcam.generate(x, pred)
    cam_big = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224))) / 255.0
    heatmap = plt.cm.jet(cam_big)[:, :, :3]
    original = np.array(image.resize((224, 224))) / 255.0
    overlay = 0.55 * heatmap + 0.45 * original
    return overlay, pred, conf


In [ ]:
# Grad-CAM comparison - ek-i image e 2 ta model kothay focus koreche pasapasi dekhi
# Prottek class theke 2 ta kore test image: 1st NORMAL, 2nd NORMAL, 3rd PNEUMONIA, 4th PNEUMONIA
picked = []
for label in range(2):
    count = 0
    for path, y in test_items:
        if y == label and count < 2:
            picked.append((path, y))
            count += 1

dn_layer = list(densenet.features.children())[-2]
rn_layer = resnet.layer4[-1]

fig, axes = plt.subplots(4, 3, figsize=(10, 13))
for row, (path, label) in enumerate(picked):
    image = Image.open(path).convert('RGB')

    dn_overlay, dn_pred, dn_conf = cam_overlay(densenet, dn_layer, image)
    rn_overlay, rn_pred, rn_conf = cam_overlay(resnet, rn_layer, image)
    original = np.array(image.resize((224, 224))) / 255.0

    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f'Original: {classes[label]}')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(dn_overlay)
    axes[row, 1].set_title(f'DenseNet121: {classes[dn_pred]} ({dn_conf:.1f}%)')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(rn_overlay)
    axes[row, 2].set_title(f'ResNet50: {classes[rn_pred]} ({rn_conf:.1f}%)')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'gradcam_comparison.png'), dpi=300)
plt.show()
print('Grad-CAM comparison save hoyeche')


In [ ]:
# McNemar test - 2 ta model er difference ta statistically significant kina
# b = DenseNet thik, ResNet vul | c = DenseNet vul, ResNet thik
label = results['DenseNet121']['label']
dn_right = results['DenseNet121']['pred'] == label
rn_right = results['ResNet50']['pred'] == label
b = int(np.sum(dn_right & ~rn_right))
c = int(np.sum(~dn_right & rn_right))
if b + c > 0:
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    p_value = math.erfc(math.sqrt(chi2 / 2))
else:
    chi2, p_value = 0.0, 1.0
print(f'McNemar: b={b}, c={c}, chi2={chi2:.3f}, p={p_value:.4f}')
print('p < 0.05 hole difference statistically significant')


In [ ]:
# Sob result Drive er Result folder e save kori
lines = []
lines.append('Pneumonia Detection - Comparative Study: DenseNet121 vs ResNet50')
lines.append('Dataset: Chest X-Ray Images (Pneumonia) - Kermany et al.')
lines.append('Protocol: train+val pool theke seed 42, 12% validation split; test set (624) ekbar evaluate')
lines.append('')
for name in ['DenseNet121', 'ResNet50']:
    r = results[name]
    cm = r['cm']
    tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    lines.append(f'--- {name} ---')
    lines.append(f"Accuracy    : {r['acc'] * 100:.2f}%")
    lines.append(f"Macro F1    : {r['f1'] * 100:.2f}%")
    lines.append(f'ROC-AUC     : {r["auc"]:.4f}')
    lines.append(f'PR-AUC      : {r["ap"]:.4f}')
    lines.append(f'Sensitivity : {tp / (tp + fn) * 100:.2f}%')
    lines.append(f'Specificity : {tn / (tn + fp) * 100:.2f}%')
    for i, cls in enumerate(classes):
        tpi = cm[i][i]
        precision = tpi / cm[:, i].sum() * 100
        recall = tpi / cm[i, :].sum() * 100
        cf1 = 2 * precision * recall / (precision + recall)
        lines.append(f'{cls:10s} Precision={precision:.2f}  Recall={recall:.2f}  F1={cf1:.2f}  Total={cm[i, :].sum()}')
    lines.append('Confusion Matrix:')
    lines.append(str(cm))
    lines.append(f'Errors: {int(cm.sum() - np.trace(cm))} / {int(cm.sum())}')
    lines.append('')
lines.append(f'McNemar test: b={b}, c={c}, chi2={chi2:.3f}, p={p_value:.4f}')
lines.append('(b = DenseNet thik kintu ResNet vul, c = DenseNet vul kintu ResNet thik)')

with open(os.path.join(RESULT_FOLDER, 'results.txt'), 'w') as f:
    f.write('\n'.join(lines))

shutil.copy('best_DenseNet121.pth', os.path.join(RESULT_FOLDER, 'best_DenseNet121.pth'))
shutil.copy('best_ResNet50.pth', os.path.join(RESULT_FOLDER, 'best_ResNet50.pth'))

print('Save hoyeche:')
for name in sorted(os.listdir(RESULT_FOLDER)):
    print(' -', name)
